In [1]:
%pip -q install pyspark pandas numpy scikit-learn imbalanced-learn pyarrow joblib

from google.colab import drive
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

drive.mount('/content/drive', force_remount=False)

spark = SparkSession.builder.appName('OlistChurnSMOTENCPreparation').getOrCreate()

data_dir = Path('/content/drive/MyDrive/BigData/DA-BigData/code/data')
processed_dir = data_dir / 'processed'
selected_feature_dir = processed_dir / 'olist_customer_churn_features_selected'
selected_feature_metadata_path = processed_dir / 'olist_customer_churn_features_selected_metadata.json'

print('Data directory           :', data_dir)
print('Processed directory      :', processed_dir)
print('Selected feature dataset :', selected_feature_dir)
print('Feature metadata         :', selected_feature_metadata_path)


Mounted at /content/drive
Data directory           : /content/drive/MyDrive/BigData/DA-BigData/code/data
Processed directory      : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed
Selected feature dataset : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_customer_churn_features_selected
Feature metadata         : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_customer_churn_features_selected_metadata.json


In [2]:
data = spark.read.option('header', True).csv(str(selected_feature_dir))

if 'temporal_split' not in data.columns:
    raise ValueError('Không tìm thấy cột temporal_split')

train_df = data.filter(F.col('temporal_split') == 'train').cache()
test_df = data.filter(F.col('temporal_split') == 'test').cache()
embargo_df = data.filter(F.col('temporal_split') == 'embargo').cache()

print('Train   :', train_df.count(), 'rows')
print('Test    :', test_df.count(), 'rows')
print('Embargo :', embargo_df.count(), 'rows')
print('Total   :', train_df.count() + test_df.count() + embargo_df.count(), 'rows')

print('\nTrain churn distribution')
train_df.groupBy('churn_180').count().orderBy('churn_180').show()

print('Test churn distribution')
test_df.groupBy('churn_180').count().orderBy('churn_180').show()

print('Embargo churn distribution')
embargo_df.groupBy('churn_180').count().orderBy('churn_180').show()


Train   : 18233 rows
Test    : 7608 rows
Embargo : 32531 rows
Total   : 58372 rows

Train churn distribution
+---------+-----+
|churn_180|count|
+---------+-----+
|        0|  629|
|        1|17604|
+---------+-----+

Test churn distribution
+---------+-----+
|churn_180|count|
+---------+-----+
|        0|  936|
|        1| 6672|
+---------+-----+

Embargo churn distribution
+---------+-----+
|churn_180|count|
+---------+-----+
|        0| 1040|
|        1|31491|
+---------+-----+



In [3]:
import json
from pyspark.sql import functions as F

metadata_path = selected_feature_metadata_path

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Không tìm thấy metadata của Step 05: {metadata_path}"
    )

with open(metadata_path, "r", encoding="utf-8") as f:
    feature_metadata = json.load(f)

feature_columns = feature_metadata.get("final_features", [])

if not feature_columns:
    raise ValueError("Không tìm thấy final_features trong metadata Step 05")

missing_in_train = [c for c in feature_columns if c not in train_df.columns]
missing_in_test = [c for c in feature_columns if c not in test_df.columns]
missing_in_embargo = [c for c in feature_columns if c not in embargo_df.columns]

if missing_in_train:
    raise ValueError("TRAIN thiếu FINAL_FEATURES:\n" + "\n".join(missing_in_train))
if missing_in_test:
    raise ValueError("TEST thiếu FINAL_FEATURES:\n" + "\n".join(missing_in_test))
if missing_in_embargo:
    raise ValueError("EMBARGO thiếu FINAL_FEATURES:\n" + "\n".join(missing_in_embargo))

print("=" * 80)
print("FINAL FEATURES ĐỌC TỪ STEP 05")
print("=" * 80)
print("Số FINAL_FEATURES:", len(feature_columns))
for i, c in enumerate(feature_columns, start=1):
    print(f"{i:02d}. {c}")

def cast_numeric(df):
    return df.select(
        *[
            F.expr(f"try_cast(`{c}` as double)").alias(c)
            if c in feature_columns else F.col(c)
            for c in df.columns
        ]
    )

train_numeric = cast_numeric(train_df)
test_numeric = cast_numeric(test_df)
embargo_numeric = cast_numeric(embargo_df)

def missing_summary(df, split_name):
    n = df.count()
    if n == 0:
        raise ValueError(f"{split_name} không có dữ liệu")
    aggs = []
    for c in feature_columns:
        missing = F.col(c).isNull() | F.isnan(F.col(c))
        aggs.append(F.sum(F.when(missing, 1).otherwise(0)).alias(c))
    result = df.agg(*aggs).first().asDict()
    return [
        (split_name, c, n, int(result[c] or 0), round(100 * int(result[c] or 0) / n, 4))
        for c in feature_columns
    ]

missing_rows = missing_summary(train_numeric, "train") + missing_summary(test_numeric, "test")
missing_df = spark.createDataFrame(
    missing_rows,
    "split string, feature string, total_rows long, missing_rows long, missing_pct double"
)

print("\n" + "=" * 80)
print(f"MISSING VALUE TRÊN {len(feature_columns)} FINAL FEATURES")
print("=" * 80)
missing_df.orderBy(F.desc("missing_pct"), "feature", "split").show(100, truncate=False)

print("=" * 80)
print("FEATURE CÓ MISSING TRONG TRAIN")
print("=" * 80)
train_missing_features = [
    r["feature"]
    for r in (
        missing_df
        .filter((F.col("split") == "train") & (F.col("missing_rows") > 0))
        .select("feature")
        .collect()
    )
]
print(train_missing_features)
print("Số feature cần impute:", len(train_missing_features))
print("\n✓ Feature list lấy trực tiếp từ Step 05")
print("✓ Không còn sử dụng feature list cũ")
print("✓ TEST chỉ được kiểm tra missing, không dùng để fit preprocessing")


FINAL FEATURES ĐỌC TỪ STEP 05
Số FINAL_FEATURES: 27
01. customer_tenure_days
02. orders_last_30d
03. orders_last_180d
04. monetary_total
05. freight_total
06. total_items
07. sum_distinct_products_per_order
08. sum_distinct_sellers_per_order
09. sum_categories_per_order
10. items_missing_category
11. mean_order_product_weight_g
12. delivered_orders_known
13. late_orders_known
14. total_payment_lines
15. max_installments
16. credit_card_orders
17. orders_payment_not_matched
18. current_item_count
19. current_distinct_products
20. current_distinct_sellers
21. current_categories
22. current_used_credit_card
23. unique_categories
24. avg_product_price
25. payment_type_count
26. known_review_count
27. low_score_review_count

MISSING VALUE TRÊN 27 FINAL FEATURES
+-----+-------------------------------+----------+------------+-----------+
|split|feature                        |total_rows|missing_rows|missing_pct|
+-----+-------------------------------+----------+------------+-----------+
|trai

In [4]:
train_selected = train_numeric
test_selected = test_numeric
embargo_selected = embargo_numeric

print("=" * 80)
print("SỬ DỤNG FINAL_FEATURES TỪ STEP 05")
print("=" * 80)
print("Không loại thêm feature ở Step 06.")
print("Các feature missing quá cao / constant / redundant đã được xử lý ở Step 05.")
print("\nSố FINAL_FEATURES:", len(feature_columns))
print("Train   :", train_selected.count())
print("Test    :", test_selected.count())
print("Embargo :", embargo_selected.count())
print("\n✓ Step 06 không hard-code loại feature cũ")


SỬ DỤNG FINAL_FEATURES TỪ STEP 05
Không loại thêm feature ở Step 06.
Các feature missing quá cao / constant / redundant đã được xử lý ở Step 05.

Số FINAL_FEATURES: 27
Train   : 18233
Test    : 7608
Embargo : 32531

✓ Step 06 không hard-code loại feature cũ


In [5]:
from pyspark.ml.feature import Imputer
from pyspark.sql import functions as F

features_to_impute = []
for c in feature_columns:
    missing_count = train_selected.filter(F.col(c).isNull() | F.isnan(F.col(c))).count()
    if missing_count > 0:
        features_to_impute.append(c)

print("Feature cần impute:", features_to_impute)

if features_to_impute:
    output_cols = [f"{c}__imputed" for c in features_to_impute]
    imputer = Imputer(strategy="median", inputCols=features_to_impute, outputCols=output_cols)
    imputer_model = imputer.fit(train_selected)

    def apply_imputer(df):
        temp = imputer_model.transform(df)
        return temp.select(
            *[
                F.col(f"{c}__imputed").alias(c) if c in features_to_impute else F.col(c)
                for c in df.columns
            ]
        )

    train_imputed = apply_imputer(train_selected)
    test_imputed = apply_imputer(test_selected)
    embargo_imputed = apply_imputer(embargo_selected)

    print("=" * 80)
    print("MEDIAN ĐƯỢC HỌC CHỈ TỪ TRAIN")
    print("=" * 80)
    imputer_model.surrogateDF.show(truncate=False)
else:
    train_imputed = train_selected
    test_imputed = test_selected
    embargo_imputed = embargo_selected

def count_missing(df):
    result = {}
    for c in feature_columns:
        n = df.filter(F.col(c).isNull() | F.isnan(F.col(c))).count()
        if n > 0:
            result[c] = n
    return result

print("=" * 80)
print("KIỂM TRA SAU IMPUTATION")
print("=" * 80)
print("Missing còn lại trong TRAIN:", count_missing(train_imputed))
print("Missing còn lại trong TEST :", count_missing(test_imputed))
print("Train rows:", train_imputed.count())
print("Test rows :", test_imputed.count())
print("Số feature:", len(feature_columns))
print("\n✓ Imputer fit TRAIN; TEST/EMBARGO chỉ transform")


Feature cần impute: ['monetary_total', 'mean_order_product_weight_g', 'max_installments', 'current_used_credit_card', 'payment_type_count']
MEDIAN ĐƯỢC HỌC CHỈ TỪ TRAIN
+--------------+---------------------------+----------------+------------------------+------------------+
|monetary_total|mean_order_product_weight_g|max_installments|current_used_credit_card|payment_type_count|
+--------------+---------------------------+----------------+------------------------+------------------+
|104.7         |800.0                      |2.0             |1.0                     |1.0               |
+--------------+---------------------------+----------------+------------------------+------------------+

KIỂM TRA SAU IMPUTATION
Missing còn lại trong TRAIN: {}
Missing còn lại trong TEST : {}
Train rows: 18233
Test rows : 7608
Số feature: 27

✓ Imputer fit TRAIN; TEST/EMBARGO chỉ transform


In [6]:
import numpy as np
import pandas as pd

if "feature_columns" not in globals():
    raise RuntimeError("Chưa có feature_columns. Hãy chạy cell đọc FINAL_FEATURES từ Step 05 trước.")

continuous_features = [
    c for c in [
        "customer_tenure_days",
        "monetary_total",
        "freight_total",
        "mean_order_product_weight_g",
        "avg_product_price"
    ]
    if c in feature_columns
]

discrete_features = [c for c in feature_columns if c not in continuous_features]

overlap = sorted(set(continuous_features) & set(discrete_features))
missing = sorted(set(feature_columns) - set(continuous_features + discrete_features))
extra = sorted(set(continuous_features + discrete_features) - set(feature_columns))

if overlap:
    raise ValueError(f"Feature bị trùng Continuous/Discrete: {overlap}")
if missing or extra:
    raise ValueError(f"Feature không khớp. Thiếu: {missing}, dư: {extra}")

continuous_indices = [feature_columns.index(c) for c in continuous_features]
discrete_indices = [feature_columns.index(c) for c in discrete_features]
categorical_indices = discrete_indices

print("=" * 80)
print("PHÂN LOẠI FINAL FEATURES")
print("=" * 80)
print("Tổng FINAL_FEATURES:", len(feature_columns))
print("Continuous:", len(continuous_features))
print("Discrete/Categorical:", len(discrete_features))

print("\nCONTINUOUS FEATURES")
for i, c in enumerate(continuous_features, start=1):
    print(f"{i:02d}. {c}")

print("\nDISCRETE/CATEGORICAL FEATURES")
for i, c in enumerate(discrete_features, start=1):
    print(f"{i:02d}. {c}")

print("\nSMOTENC CATEGORICAL INDICES:")
print(categorical_indices)

assert len(continuous_features) + len(discrete_features) == len(feature_columns)
assert set(continuous_features).isdisjoint(set(discrete_features))
assert set(continuous_features + discrete_features) == set(feature_columns)
print("\n✓ Phân loại feature khớp hoàn toàn với FINAL_FEATURES của Step 05")


PHÂN LOẠI FINAL FEATURES
Tổng FINAL_FEATURES: 27
Continuous: 5
Discrete/Categorical: 22

CONTINUOUS FEATURES
01. customer_tenure_days
02. monetary_total
03. freight_total
04. mean_order_product_weight_g
05. avg_product_price

DISCRETE/CATEGORICAL FEATURES
01. orders_last_30d
02. orders_last_180d
03. total_items
04. sum_distinct_products_per_order
05. sum_distinct_sellers_per_order
06. sum_categories_per_order
07. items_missing_category
08. delivered_orders_known
09. late_orders_known
10. total_payment_lines
11. max_installments
12. credit_card_orders
13. orders_payment_not_matched
14. current_item_count
15. current_distinct_products
16. current_distinct_sellers
17. current_categories
18. current_used_credit_card
19. unique_categories
20. payment_type_count
21. known_review_count
22. low_score_review_count

SMOTENC CATEGORICAL INDICES:
[1, 2, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26]

✓ Phân loại feature khớp hoàn toàn với FINAL_FEATURES của Step 05


In [7]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

train_pd = (
    train_imputed
    .select(*feature_columns, F.col("churn_180").cast("int").alias("churn_180"))
    .toPandas()
)

test_pd = (
    test_imputed
    .select(*feature_columns, F.col("churn_180").cast("int").alias("churn_180"))
    .toPandas()
)

for c in feature_columns:
    train_pd[c] = pd.to_numeric(train_pd[c], errors="coerce")
    test_pd[c] = pd.to_numeric(test_pd[c], errors="coerce")

if train_pd[feature_columns].isna().any().any():
    raise ValueError("TRAIN còn NaN sau imputation")
if test_pd[feature_columns].isna().any().any():
    raise ValueError("TEST còn NaN sau imputation")
if np.isinf(train_pd[feature_columns].to_numpy(dtype=float)).any():
    raise ValueError("TRAIN còn Inf")
if np.isinf(test_pd[feature_columns].to_numpy(dtype=float)).any():
    raise ValueError("TEST còn Inf")

X_train_raw = train_pd[feature_columns].astype(float).copy()
X_test_raw = test_pd[feature_columns].astype(float).copy()
y_train_smotenc = train_pd["churn_180"].astype(int).to_numpy()
y_test_smotenc = test_pd["churn_180"].astype(int).to_numpy()

scaler_continuous = StandardScaler()
X_train_smotenc = X_train_raw.copy()
X_test_smotenc = X_test_raw.copy()

if continuous_features:
    X_train_smotenc.loc[:, continuous_features] = scaler_continuous.fit_transform(
        X_train_raw[continuous_features]
    )
    X_test_smotenc.loc[:, continuous_features] = scaler_continuous.transform(
        X_test_raw[continuous_features]
    )

X_train_smotenc = X_train_smotenc[feature_columns]
X_test_smotenc = X_test_smotenc[feature_columns]

if X_train_smotenc.shape[1] != len(feature_columns):
    raise ValueError("Sai số feature TRAIN sau preprocessing")
if X_test_smotenc.shape[1] != len(feature_columns):
    raise ValueError("Sai số feature TEST sau preprocessing")

print("=" * 90)
print("DỮ LIỆU ĐẦU VÀO CHO SMOTENC")
print("=" * 90)
print("X_train_smotenc:", X_train_smotenc.shape)
print("y_train_smotenc:", y_train_smotenc.shape)
print("X_test_smotenc :", X_test_smotenc.shape)
print("y_test_smotenc :", y_test_smotenc.shape)
print("Số feature     :", len(feature_columns))
print("Continuous scaled:", len(continuous_features))
print("Discrete giữ nguyên:", len(discrete_features))
print("\n✓ StandardScaler fit chỉ trên TRAIN continuous features")
print("✓ TEST chỉ transform")
print("✓ X_train_smotenc đã có đúng FINAL_FEATURES từ Step 05")


DỮ LIỆU ĐẦU VÀO CHO SMOTENC
X_train_smotenc: (18233, 27)
y_train_smotenc: (18233,)
X_test_smotenc : (7608, 27)
y_test_smotenc : (7608,)
Số feature     : 27
Continuous scaled: 5
Discrete giữ nguyên: 22

✓ StandardScaler fit chỉ trên TRAIN continuous features
✓ TEST chỉ transform
✓ X_train_smotenc đã có đúng FINAL_FEATURES từ Step 05


In [8]:
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTENC

required_vars = [
    "X_train_smotenc",
    "y_train_smotenc",
    "X_test_smotenc",
    "y_test_smotenc",
    "feature_columns",
    "continuous_features",
    "discrete_features",
    "categorical_indices",
    "scaler_continuous"
]

missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    raise ValueError(f"Thiếu biến từ cell trước: {missing_vars}")

if X_train_smotenc.shape[1] != len(feature_columns):
    raise ValueError(
        f"TRAIN có {X_train_smotenc.shape[1]} feature, "
        f"nhưng feature_columns có {len(feature_columns)}"
    )

if X_test_smotenc.shape[1] != len(feature_columns):
    raise ValueError(
        f"TEST có {X_test_smotenc.shape[1]} feature, "
        f"nhưng feature_columns có {len(feature_columns)}"
    )

if X_train_smotenc.isna().any().any():
    raise ValueError("TRAIN còn NaN")

if X_test_smotenc.isna().any().any():
    raise ValueError("TEST còn NaN")

if np.isinf(X_train_smotenc.to_numpy()).any():
    raise ValueError("TRAIN còn Inf")

if np.isinf(X_test_smotenc.to_numpy()).any():
    raise ValueError("TEST còn Inf")

train_class_before = (
    pd.Series(y_train_smotenc)
    .value_counts()
    .sort_index()
)

test_class_before = (
    pd.Series(y_test_smotenc)
    .value_counts()
    .sort_index()
)

print("=" * 90)
print("PHÂN BỐ TRAIN TRƯỚC SMOTENC")
print("=" * 90)

for cls, count in train_class_before.items():
    print(
        f"churn_180 = {cls}: {count:,} "
        f"({100 * count / len(y_train_smotenc):.2f}%)"
    )

minority_class = train_class_before.idxmin()
majority_class = train_class_before.idxmax()

print("\nMinority class :", minority_class)
print("Majority class :", majority_class)

print("\nContinuous features:")
for c in continuous_features:
    print("-", c)

print("\nDiscrete/Categorical features:")
for c in discrete_features:
    print("-", c)

print("\nCategorical indices:")
print(categorical_indices)

minority_count = int(train_class_before.min())
if minority_count < 2:
    raise ValueError("Minority class có ít hơn 2 mẫu, không thể chạy SMOTENC")

smotenc_k_neighbors = min(5, minority_count - 1)

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy=1.0,
    random_state=42,
    k_neighbors=smotenc_k_neighbors
)

X_train_smotenc_resampled, y_train_smotenc_resampled = (
    smotenc.fit_resample(
        X_train_smotenc,
        y_train_smotenc
    )
)

if not isinstance(X_train_smotenc_resampled, pd.DataFrame):
    X_train_smotenc_resampled = pd.DataFrame(
        X_train_smotenc_resampled,
        columns=feature_columns
    )
else:
    X_train_smotenc_resampled.columns = feature_columns

y_train_smotenc_resampled = pd.Series(
    np.asarray(y_train_smotenc_resampled),
    name="churn_180"
)

X_test_final = X_test_smotenc.copy()
y_test_final = pd.Series(
    np.asarray(y_test_smotenc),
    name="churn_180"
)

train_class_after = (
    y_train_smotenc_resampled
    .value_counts()
    .sort_index()
)

print("=" * 90)
print("PHÂN BỐ TRAIN SAU SMOTENC")
print("=" * 90)

for cls, count in train_class_after.items():
    print(
        f"churn_180 = {cls}: {count:,} "
        f"({100 * count / len(y_train_smotenc_resampled):.2f}%)"
    )

n_original = len(X_train_smotenc)
n_after = len(X_train_smotenc_resampled)
n_synthetic = n_after - n_original

synthetic_df = (
    X_train_smotenc_resampled
    .iloc[n_original:]
    .copy()
)

print("=" * 90)
print("KIỂM TRA KÍCH THƯỚC")
print("=" * 90)

print("X_train trước SMOTENC :", X_train_smotenc.shape)
print("y_train trước SMOTENC :", np.asarray(y_train_smotenc).shape)

print("X_train sau SMOTENC   :", X_train_smotenc_resampled.shape)
print("y_train sau SMOTENC   :", y_train_smotenc_resampled.shape)

print("Synthetic samples     :", f"{n_synthetic:,}")

print("X_test                 :", X_test_final.shape)
print("y_test                 :", y_test_final.shape)

if X_train_smotenc_resampled.shape[1] != len(feature_columns):
    raise ValueError("Sai số lượng feature sau SMOTENC")

if X_train_smotenc_resampled.isna().any().any():
    raise ValueError("Dữ liệu sau SMOTENC có NaN")

if np.isinf(X_train_smotenc_resampled.to_numpy()).any():
    raise ValueError("Dữ liệu sau SMOTENC có Inf")

invalid_category_rows = []

for c in discrete_features:
    original_values = set(
        X_train_smotenc[c]
        .dropna()
        .unique()
        .tolist()
    )

    invalid_mask = ~synthetic_df[c].isin(original_values)
    invalid_n = int(invalid_mask.sum())

    non_integer_n = int(
        (
            ~np.isclose(
                synthetic_df[c].astype(float),
                np.round(synthetic_df[c].astype(float)),
                atol=1e-8
            )
        ).sum()
    )

    invalid_category_rows.append(
        {
            "feature": c,
            "synthetic_rows": n_synthetic,
            "invalid_category_rows": invalid_n,
            "non_integer_rows": non_integer_n
        }
    )

discrete_check = pd.DataFrame(invalid_category_rows)

print("=" * 90)
print("KIỂM TRA DISCRETE FEATURE SAU SMOTENC")
print("=" * 90)

display(discrete_check)

total_invalid_categories = int(
    discrete_check["invalid_category_rows"].sum()
)

total_non_integer = int(
    discrete_check["non_integer_rows"].sum()
)

binary_features = [
    c for c in [
        "orders_payment_not_matched",
        "current_used_credit_card"
    ]
    if c in discrete_features
]

binary_rows = []

for c in binary_features:
    values = synthetic_df[c].astype(float)

    valid = (
        np.isclose(values, 0, atol=1e-8)
        | np.isclose(values, 1, atol=1e-8)
    )

    binary_rows.append(
        {
            "feature": c,
            "synthetic_rows": n_synthetic,
            "invalid_binary_rows": int((~valid).sum()),
            "invalid_binary_pct": round(
                100 * (~valid).sum() / n_synthetic,
                4
            )
        }
    )

binary_check_smotenc = pd.DataFrame(binary_rows)

print("=" * 90)
print("KIỂM TRA BINARY FEATURE SAU SMOTENC")
print("=" * 90)

display(binary_check_smotenc)

continuous_range_rows = []

for c in continuous_features:
    train_min = X_train_smotenc[c].min()
    train_max = X_train_smotenc[c].max()

    below = int(
        (synthetic_df[c] < train_min - 1e-8).sum()
    )

    above = int(
        (synthetic_df[c] > train_max + 1e-8).sum()
    )

    continuous_range_rows.append(
        {
            "feature": c,
            "train_min": train_min,
            "train_max": train_max,
            "synthetic_min": synthetic_df[c].min(),
            "synthetic_max": synthetic_df[c].max(),
            "below_train_range": below,
            "above_train_range": above
        }
    )

continuous_range_check = pd.DataFrame(
    continuous_range_rows
)

print("=" * 90)
print("KIỂM TRA CONTINUOUS FEATURE")
print("=" * 90)

display(continuous_range_check)

range_violations = int(
    continuous_range_check["below_train_range"].sum()
    + continuous_range_check["above_train_range"].sum()
)

print("=" * 90)
print("KẾT LUẬN SMOTENC")
print("=" * 90)

print("SMOTENC k_neighbors                 :", smotenc_k_neighbors)
print("Synthetic samples                  :", f"{n_synthetic:,}")
print("Discrete giá trị ngoài tập hợp gốc :", total_invalid_categories)
print("Discrete trở thành số thập phân    :", total_non_integer)
print("Continuous vượt range TRAIN        :", range_violations)

print("\nTEST:")
print("Không resampling")
print("Không thay đổi nhãn")
print("Không dùng TEST để fit SMOTENC")

if (
    total_invalid_categories == 0
    and total_non_integer == 0
    and range_violations == 0
):
    print("\nSMOTENC ĐẠT KIỂM TRA.")
    print("Dữ liệu sẵn sàng cho bước Model Training.")
else:
    print("\nSMOTENC CÒN VẤN ĐỀ.")
    print("Cần kiểm tra output trước khi train model.")

print("=" * 90)
print("BIẾN SẴN SÀNG CHO BƯỚC TIẾP THEO")
print("=" * 90)

print(
    "X_train_smotenc_resampled :",
    X_train_smotenc_resampled.shape
)

print(
    "y_train_smotenc_resampled :",
    y_train_smotenc_resampled.shape
)

print(
    "X_test_final              :",
    X_test_final.shape
)

print(
    "y_test_final              :",
    y_test_final.shape
)


PHÂN BỐ TRAIN TRƯỚC SMOTENC
churn_180 = 0: 629 (3.45%)
churn_180 = 1: 17,604 (96.55%)

Minority class : 0
Majority class : 1

Continuous features:
- customer_tenure_days
- monetary_total
- freight_total
- mean_order_product_weight_g
- avg_product_price

Discrete/Categorical features:
- orders_last_30d
- orders_last_180d
- total_items
- sum_distinct_products_per_order
- sum_distinct_sellers_per_order
- sum_categories_per_order
- items_missing_category
- delivered_orders_known
- late_orders_known
- total_payment_lines
- max_installments
- credit_card_orders
- orders_payment_not_matched
- current_item_count
- current_distinct_products
- current_distinct_sellers
- current_categories
- current_used_credit_card
- unique_categories
- payment_type_count
- known_review_count
- low_score_review_count

Categorical indices:
[1, 2, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26]
PHÂN BỐ TRAIN SAU SMOTENC
churn_180 = 0: 17,604 (50.00%)
churn_180 = 1: 17,604 (50.00%)
KIỂM T

,feature,synthetic_rows,invalid_category_rows,non_integer_rows
0,orders_last_30d,16975,0,0
1,orders_last_180d,16975,0,0
2,total_items,16975,0,0
3,sum_distinct_products_per_order,16975,0,0
4,sum_distinct_sellers_per_order,16975,0,0
5,sum_categories_per_order,16975,0,0
6,items_missing_category,16975,0,0
7,delivered_orders_known,16975,0,0
8,late_orders_known,16975,0,0
9,total_payment_lines,16975,0,0


KIỂM TRA BINARY FEATURE SAU SMOTENC


,feature,synthetic_rows,invalid_binary_rows,invalid_binary_pct
0,orders_payment_not_matched,16975,0,0.0
1,current_used_credit_card,16975,0,0.0


KIỂM TRA CONTINUOUS FEATURE


,feature,train_min,train_max,synthetic_min,synthetic_max,below_train_range,above_train_range
0,customer_tenure_days,-0.079666,41.384395,-0.079666,15.387781,0,0
1,monetary_total,-0.628970,30.910273,-0.628922,19.178705,0,0
2,freight_total,-0.730819,48.822691,-0.730819,9.035786,0,0
3,mean_order_product_weight_g,-0.544250,6.684458,-0.541622,6.684458,0,0
4,avg_product_price,-0.580775,30.695069,-0.574286,20.711692,0,0


KẾT LUẬN SMOTENC
SMOTENC k_neighbors                 : 5
Synthetic samples                  : 16,975
Discrete giá trị ngoài tập hợp gốc : 0
Discrete trở thành số thập phân    : 0
Continuous vượt range TRAIN        : 0

TEST:
Không resampling
Không thay đổi nhãn
Không dùng TEST để fit SMOTENC

SMOTENC ĐẠT KIỂM TRA.
Dữ liệu sẵn sàng cho bước Model Training.
BIẾN SẴN SÀNG CHO BƯỚC TIẾP THEO
X_train_smotenc_resampled : (35208, 27)
y_train_smotenc_resampled : (35208,)
X_test_final              : (7608, 27)
y_test_final              : (7608,)


In [9]:
import numpy as np
import pandas as pd

n_original = len(X_train_smotenc)
n_after = len(X_train_smotenc_resampled)
n_synthetic = n_after - n_original

if n_synthetic <= 0:
    raise ValueError("Không tìm thấy synthetic sample sau SMOTENC")

synthetic_mixed = (
    X_train_smotenc_resampled
    .iloc[n_original:]
    .copy()
    .reset_index(drop=True)
)

synthetic_y = np.asarray(
    y_train_smotenc_resampled
)[n_original:]

if not np.all(synthetic_y == minority_class):
    raise ValueError(
        f"Synthetic sample không hoàn toàn thuộc minority class churn_180 = {minority_class}"
    )

minority_mask = np.asarray(y_train_smotenc) == minority_class

original_non_churn_mixed = (
    X_train_smotenc
    .iloc[np.where(minority_mask)[0]]
    .copy()
    .reset_index(drop=True)
)

train_original_mixed = (
    X_train_smotenc
    .copy()
    .reset_index(drop=True)
)

synthetic_original = synthetic_mixed.copy()
original_non_churn = original_non_churn_mixed.copy()
train_original = train_original_mixed.copy()

synthetic_original[continuous_features] = (
    scaler_continuous.inverse_transform(
        synthetic_mixed[continuous_features]
    )
)

original_non_churn[continuous_features] = (
    scaler_continuous.inverse_transform(
        original_non_churn_mixed[continuous_features]
    )
)

train_original[continuous_features] = (
    scaler_continuous.inverse_transform(
        train_original_mixed[continuous_features]
    )
)

print("=" * 100)
print("1. TÁCH MẪU GỐC VÀ MẪU SYNTHETIC — SMOTENC")
print("=" * 100)

print("TRAIN gốc                  :", f"{n_original:,}")
print("Non-churn gốc              :", f"{len(original_non_churn):,}")
print("Synthetic do SMOTENC tạo   :", f"{n_synthetic:,}")
print("TRAIN sau SMOTENC          :", f"{n_after:,}")

print("\nContinuous features:")
print(continuous_features)

print("\nDiscrete features:")
print(discrete_features)

print("=" * 100)
print("2. MẪU SYNTHETIC Ở THANG ĐO GỐC")
print("=" * 100)

display(synthetic_original.head(10))

binary_features = [
    c for c in [
        "orders_payment_not_matched",
        "current_used_credit_card"
    ]
    if c in discrete_features
]

binary_rows = []

for c in binary_features:

    values = synthetic_original[c].astype(float).to_numpy()

    valid = (
        np.isclose(values, 0, atol=1e-8)
        | np.isclose(values, 1, atol=1e-8)
    )

    invalid_n = int((~valid).sum())

    binary_rows.append({
        "feature": c,
        "synthetic_rows": len(values),
        "invalid_binary_rows": invalid_n,
        "invalid_binary_pct": round(
            100 * invalid_n / len(values), 4
        ),
        "min": float(np.min(values)),
        "max": float(np.max(values))
    })

binary_check = pd.DataFrame(binary_rows)

print("=" * 100)
print("3. KIỂM TRA FEATURE NHỊ PHÂN")
print("=" * 100)

display(binary_check)

discrete_rows = []

for c in discrete_features:

    synthetic_values = (
        synthetic_original[c]
        .astype(float)
        .to_numpy()
    )

    original_values = set(
        train_original[c]
        .astype(float)
        .unique()
        .tolist()
    )

    outside_original = np.array([
        v not in original_values
        for v in synthetic_values
    ])

    non_integer = ~np.isclose(
        synthetic_values,
        np.round(synthetic_values),
        atol=1e-8
    )

    negative = synthetic_values < -1e-8

    discrete_rows.append({
        "feature": c,
        "synthetic_rows": len(synthetic_values),
        "outside_original_values": int(
            outside_original.sum()
        ),
        "non_integer_rows": int(
            non_integer.sum()
        ),
        "non_integer_pct": round(
            100 * non_integer.mean(), 4
        ),
        "negative_rows": int(
            negative.sum()
        ),
        "min": float(
            synthetic_values.min()
        ),
        "max": float(
            synthetic_values.max()
        )
    })

discrete_check = pd.DataFrame(
    discrete_rows
).sort_values(
    ["outside_original_values", "non_integer_rows"],
    ascending=False
)

print("=" * 100)
print("4. KIỂM TRA DISCRETE / COUNT FEATURE")
print("=" * 100)

display(discrete_check)

comparison_rows = []

for c in feature_columns:

    original_values = original_non_churn[c].astype(float)
    synthetic_values = synthetic_original[c].astype(float)
    train_values = train_original[c].astype(float)

    comparison_rows.append({
        "feature": c,

        "original_min":
            original_values.min(),

        "synthetic_min":
            synthetic_values.min(),

        "original_median":
            original_values.median(),

        "synthetic_median":
            synthetic_values.median(),

        "original_mean":
            original_values.mean(),

        "synthetic_mean":
            synthetic_values.mean(),

        "original_max":
            original_values.max(),

        "synthetic_max":
            synthetic_values.max(),

        "train_min":
            train_values.min(),

        "train_max":
            train_values.max()
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

print("=" * 100)
print("5. SO SÁNH NON-CHURN GỐC VỚI SYNTHETIC")
print("=" * 100)

display(comparison_df)

continuous_range_rows = []

for c in continuous_features:

    train_min = train_original[c].min()
    train_max = train_original[c].max()

    synthetic_min = synthetic_original[c].min()
    synthetic_max = synthetic_original[c].max()

    below = int(
        (
            synthetic_original[c]
            < train_min - 1e-8
        ).sum()
    )

    above = int(
        (
            synthetic_original[c]
            > train_max + 1e-8
        ).sum()
    )

    continuous_range_rows.append({
        "feature": c,
        "train_min": train_min,
        "train_max": train_max,
        "synthetic_min": synthetic_min,
        "synthetic_max": synthetic_max,
        "below_train_range": below,
        "above_train_range": above
    })

continuous_range_check = pd.DataFrame(
    continuous_range_rows
)

print("=" * 100)
print("6. KIỂM TRA PHẠM VI CONTINUOUS FEATURE")
print("=" * 100)

display(continuous_range_check)

binary_invalid_total = (
    int(binary_check["invalid_binary_rows"].sum())
    if len(binary_check) > 0
    else 0
)

discrete_invalid_total = int(
    discrete_check[
        "outside_original_values"
    ].sum()
)

non_integer_total = int(
    discrete_check[
        "non_integer_rows"
    ].sum()
)

range_violations = int(
    continuous_range_check[
        "below_train_range"
    ].sum()
    +
    continuous_range_check[
        "above_train_range"
    ].sum()
)

print("=" * 100)
print("7. KẾT LUẬN CHẤT LƯỢNG SMOTENC")
print("=" * 100)

print(
    "Synthetic samples                  :",
    f"{n_synthetic:,}"
)

print(
    "Binary không còn đúng 0/1          :",
    binary_invalid_total
)

print(
    "Discrete ngoài tập giá trị TRAIN   :",
    discrete_invalid_total
)

print(
    "Discrete / count thành số thập phân:",
    non_integer_total
)

print(
    "Continuous vượt range TRAIN        :",
    range_violations
)

print("\nTEST:")
print("Số dòng:", len(X_test_final))
print("Không resampling")
print("Không dùng TEST khi fit SMOTENC")

print("=" * 100)

if (
    binary_invalid_total == 0
    and discrete_invalid_total == 0
    and non_integer_total == 0
    and range_violations == 0
):

    print("SMOTENC ĐẠT TẤT CẢ KIỂM TRA")

    print(
        "→ Binary vẫn hợp lệ"
    )

    print(
        "→ Count/discrete vẫn giữ giá trị rời rạc"
    )

    print(
        "→ Continuous không vượt phạm vi TRAIN"
    )

    print(
        "→ Có thể chuyển sang Model Training"
    )

else:

    print("SMOTENC CHƯA ĐẠT")

    print(
        "→ Cần xem bảng kiểm tra phía trên "
        "trước khi train model"
    )


1. TÁCH MẪU GỐC VÀ MẪU SYNTHETIC — SMOTENC
TRAIN gốc                  : 18,233
Non-churn gốc              : 629
Synthetic do SMOTENC tạo   : 16,975
TRAIN sau SMOTENC          : 35,208

Continuous features:
['customer_tenure_days', 'monetary_total', 'freight_total', 'mean_order_product_weight_g', 'avg_product_price']

Discrete features:
['orders_last_30d', 'orders_last_180d', 'total_items', 'sum_distinct_products_per_order', 'sum_distinct_sellers_per_order', 'sum_categories_per_order', 'items_missing_category', 'delivered_orders_known', 'late_orders_known', 'total_payment_lines', 'max_installments', 'credit_card_orders', 'orders_payment_not_matched', 'current_item_count', 'current_distinct_products', 'current_distinct_sellers', 'current_categories', 'current_used_credit_card', 'unique_categories', 'payment_type_count', 'known_review_count', 'low_score_review_count']
2. MẪU SYNTHETIC Ở THANG ĐO GỐC


,customer_tenure_days,orders_last_30d,orders_last_180d,monetary_total,freight_total,total_items,sum_distinct_products_per_order,sum_distinct_sellers_per_order,sum_categories_per_order,items_missing_category,...,current_item_count,current_distinct_products,current_distinct_sellers,current_categories,current_used_credit_card,unique_categories,avg_product_price,payment_type_count,known_review_count,low_score_review_count
0,0.000000,1.0,1.0,106.226845,15.385918,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,90.840927,1.0,0.0,0.0
1,0.000000,1.0,1.0,35.220776,19.499248,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,15.721528,1.0,0.0,0.0
2,0.000000,1.0,1.0,99.222228,15.862228,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,0.0,1.0,83.360000,1.0,0.0,0.0
3,0.000000,1.0,1.0,54.796196,14.995115,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,0.0,1.0,39.801081,1.0,0.0,0.0
4,0.001762,2.0,2.0,28.278548,17.440000,2.0,2.0,2.0,2.0,0.0,...,1.0,1.0,1.0,1.0,0.0,1.0,5.419274,1.0,0.0,0.0
5,0.000000,1.0,1.0,99.200537,17.478749,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,81.721788,1.0,0.0,0.0
6,0.000000,1.0,1.0,156.409643,16.147236,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,140.262407,1.0,0.0,0.0
7,0.000000,1.0,1.0,155.423652,33.639677,2.0,1.0,1.0,1.0,0.0,...,2.0,1.0,1.0,1.0,1.0,1.0,60.891987,1.0,0.0,0.0
8,0.000000,1.0,1.0,152.148972,18.919512,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,133.229460,1.0,0.0,0.0
9,0.000000,1.0,1.0,36.305941,15.100000,1.0,1.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,0.0,1.0,21.205941,1.0,0.0,0.0


3. KIỂM TRA FEATURE NHỊ PHÂN


,feature,synthetic_rows,invalid_binary_rows,invalid_binary_pct,min,max
0,orders_payment_not_matched,16975,0,0.0,0.0,0.0
1,current_used_credit_card,16975,0,0.0,0.0,1.0


4. KIỂM TRA DISCRETE / COUNT FEATURE


...

5. SO SÁNH NON-CHURN GỐC VỚI SYNTHETIC


,feature,original_min,synthetic_min,original_median,synthetic_median,original_mean,synthetic_mean,original_max,synthetic_max,train_min,train_max
0,customer_tenure_days,0.00,0.000000,0.00,0.000000,1.762881,1.682776,110.968125,110.709152,0.000000,296.781435
1,orders_last_30d,1.00,1.000000,1.00,1.000000,1.079491,1.059499,5.000000,3.000000,1.000000,6.000000
2,orders_last_180d,1.00,1.000000,1.00,1.000000,1.104928,1.080118,5.000000,3.000000,1.000000,6.000000
3,monetary_total,11.62,11.631406,102.29,100.516496,160.404229,146.604403,4950.340000,4759.553236,11.620000,7571.630000
4,freight_total,7.78,7.780000,17.53,17.264687,25.054785,23.844516,204.130000,203.790044,7.780000,1002.290000
5,total_items,1.00,1.000000,1.00,1.000000,1.322734,1.241060,6.000000,4.000000,1.000000,21.000000
6,sum_distinct_products_per_order,1.00,1.000000,1.00,1.000000,1.200318,1.141915,5.000000,4.000000,1.000000,7.000000
7,sum_distinct_sellers_per_order,1.00,1.000000,1.00,1.000000,1.125596,1.088601,5.000000,3.000000,1.000000,6.000000
8,sum_categories_per_order,0.00,0.000000,1.00,1.000000,1.108108,1.076053,5.000000,3.000000,0.000000,6.000000
9,items_missing_category,0.00,0.000000,0.00,0.000000,0.025437,0.010898,3.000000,1.000000,0.000000,6.000000


6. KIỂM TRA PHẠM VI CONTINUOUS FEATURE


,feature,train_min,train_max,synthetic_min,synthetic_max,below_train_range,above_train_range
0,customer_tenure_days,0.000000,296.781435,0.000000,110.709152,0,0
1,monetary_total,11.620000,7571.630000,11.631406,4759.553236,0,0
2,freight_total,7.780000,1002.290000,7.780000,203.790044,0,0
3,mean_order_product_weight_g,50.000000,30000.000000,60.888131,30000.000000,0,0
4,avg_product_price,1.514286,6735.000000,2.911406,4585.643945,0,0


7. KẾT LUẬN CHẤT LƯỢNG SMOTENC
Synthetic samples                  : 16,975
Binary không còn đúng 0/1          : 0
Discrete ngoài tập giá trị TRAIN   : 0
Discrete / count thành số thập phân: 0
Continuous vượt range TRAIN        : 0

TEST:
Số dòng: 7608
Không resampling
Không dùng TEST khi fit SMOTENC
SMOTENC ĐẠT TẤT CẢ KIỂM TRA
→ Binary vẫn hợp lệ
→ Count/discrete vẫn giữ giá trị rời rạc
→ Continuous không vượt phạm vi TRAIN
→ Có thể chuyển sang Model Training


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json

save_dir = Path(
    "/content/drive/MyDrive/BigData/DA-BigData/code/data/processed/"
    "olist_churn_smotenc_ready"
)

save_dir.mkdir(parents=True, exist_ok=True)

train_final_df = X_train_smotenc_resampled.copy()
train_final_df["churn_180"] = np.asarray(
    y_train_smotenc_resampled
).astype(int)

train_final_df["is_synthetic"] = np.concatenate([
    np.zeros(len(X_train_smotenc), dtype=int),
    np.ones(len(train_final_df) - len(X_train_smotenc), dtype=int)
])

test_final_df = X_test_final.copy()
test_final_df["churn_180"] = np.asarray(
    y_test_final
).astype(int)

test_final_df["is_synthetic"] = 0

train_path = save_dir / "train_smotenc.parquet"
test_path = save_dir / "test_original.parquet"
scaler_path = save_dir / "continuous_scaler.joblib"
metadata_path = save_dir / "metadata.json"

train_final_df.to_parquet(
    train_path,
    index=False
)

test_final_df.to_parquet(
    test_path,
    index=False
)

joblib.dump(
    scaler_continuous,
    scaler_path
)

metadata = {
    "target": "churn_180",
    "n_features": len(feature_columns),
    "feature_columns": feature_columns,
    "continuous_features": continuous_features,
    "discrete_features": discrete_features,
    "smotenc_sampling_strategy": 1.0,
    "smotenc_random_state": 42,
    "smotenc_k_neighbors": int(smotenc_k_neighbors),
    "train_original_rows": int(len(X_train_smotenc)),
    "train_after_smotenc_rows": int(len(train_final_df)),
    "synthetic_rows": int(
        len(train_final_df) - len(X_train_smotenc)
    ),
    "test_rows": int(len(test_final_df))
}

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 90)
print("ĐÃ LƯU DATASET CHO MODEL TRAINING")
print("=" * 90)

print("TRAIN :", train_path)
print("TEST  :", test_path)
print("Scaler:", scaler_path)
print("Meta  :", metadata_path)

print("\nTRAIN shape:", train_final_df.shape)
print("TEST shape :", test_final_df.shape)

print("\nPhân bố TRAIN:")
print(
    train_final_df["churn_180"]
    .value_counts()
    .sort_index()
)

print("\nSynthetic:")
print(
    train_final_df["is_synthetic"]
    .value_counts()
    .sort_index()
)


ĐÃ LƯU DATASET CHO MODEL TRAINING
TRAIN : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_churn_smotenc_ready/train_smotenc.parquet
TEST  : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_churn_smotenc_ready/test_original.parquet
Scaler: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_churn_smotenc_ready/continuous_scaler.joblib
Meta  : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_churn_smotenc_ready/metadata.json

TRAIN shape: (35208, 29)
TEST shape : (7608, 29)

Phân bố TRAIN:
churn_180
0    17604
1    17604
Name: count, dtype: int64

Synthetic:
is_synthetic
0    18233
1    16975
Name: count, dtype: int64
